# Evaluating agents with metrics

The environment tells you what an agent *did*; the metrics module tells you how *good* it was.
This notebook compares three agents on `case30` along the measures used in the paper:
line loading (N-0 and N-1), how many lines are overloaded, how much switching the agent needed,
and the reward it collected.

Two pieces do the work, both in `pandapower_env/metrics/`:

- **`MetricRegistry`** (`evaluation_metrics.py`) is the catalog. Every metric is a function
  `(step: StepData) -> float` registered under a name, so a metric is selected by string.
- **`EvaluateMetrics`** (`metric_utils.py`) is the runner. Given an `env_config` and a list of
  actions it builds a fresh `PPTopoGym`, replays the actions, and calls the selected metrics after
  every step. It returns one row per timestep and one row per aggregation.

Scoring is therefore *separate* from rolling out an agent: first let the agent play and keep its
`env.log_actions`, then replay that list through `EvaluateMetrics`. The replay is deterministic
given the same `start_index`, and it keeps the expensive metrics (N-1 solves a whole contingency
sweep per step) out of the agent's own loop.


In [ ]:
from __future__ import annotations

import copy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from gymnasium import spaces

from pandapower_env.agents.base_agents import BaseAgent, BaseGreedyAgent
from pandapower_env.agents.benchmark_agents import DoNothingAgent, GreedyAgent, RandomAgent
from pandapower_env.data.example_configs import config_case30
from pandapower_env.environments.simulation_env import PPTopoGym
from pandapower_env.metrics.evaluation_metrics import MetricRegistry
from pandapower_env.metrics.metric_utils import EvaluateMetrics, StepData


## The metric catalog

`MetricRegistry.METRICS` maps a name to the function computing it. Anything listed here can be
requested by name in `metric_keys`; passing `None` runs all of them.


In [ ]:
catalog = pd.DataFrame(
    [(name, (fn.__doc__ or "").strip().splitlines()[0]) for name, fn in sorted(MetricRegistry.METRICS.items())],
    columns=["metric", "description"],
)
catalog.style.hide(axis="index")


## The grid

`config_case30()` scales the grid until lines overload, creates the double-busbar substations and
generates the verified action space. It is expensive, so it is built once here and deep-copied
wherever a fresh environment is needed.


In [ ]:
env_config = config_case30()
n_actions = len(env_config["action_space"])
print(f"{len(env_config['net'].line)} lines, {len(env_config['net'].multi_bb_substation)} double-busbar substations")
print(f"{n_actions} verified actions")


## Rolling out the agents

`START_INDEX` picks a stressed part of the timeseries — around timestep 1198 the do-nothing grid
runs into an overload, which is what the agents are supposed to fix.

`N_STEPS` is short on purpose: the greedy agent simulates every one of the ~290 actions per step,
so it costs a few seconds per timestep. The figures in `notebooks/plots` come from a full
96-step episode (`N_STEPS = 96`).


In [ ]:
START_INDEX = 1198
N_STEPS = 12


def collect_actions(agent: BaseAgent, n_steps: int = N_STEPS) -> list[int]:
    """Let an agent play its own environment and return the actions it took.

    :param agent: any agent exposing ``act``; greedy agents additionally need the ``info`` dict.
    :param n_steps: number of timesteps to play, starting at ``START_INDEX``.
    :return: the action sequence, ready to be replayed by :class:`EvaluateMetrics`.
    """
    env = PPTopoGym(copy.deepcopy(env_config))
    observation, info = env.reset(options={"index": START_INDEX})

    for _ in range(n_steps):
        action = agent.act(observation, info) if isinstance(agent, BaseGreedyAgent) else agent.act(observation)
        observation, _, terminated, truncated, info = env.step(int(action))
        if terminated or truncated:
            break

    return [int(action) for action in env.log_actions]


In [ ]:
action_space = spaces.Discrete(n_actions)
random_space = spaces.Discrete(n_actions)
random_space.seed(0)

agents = {
    "DoNothing": DoNothingAgent(action_space),
    "Random": RandomAgent(random_space),
    "Greedy": GreedyAgent(action_space, copy.deepcopy(env_config), n_workers=8),
}

taken_actions = {name: collect_actions(agent) for name, agent in agents.items()}
for name, actions in taken_actions.items():
    print(f"{name:10s} {actions}")


## Scoring the action sequences

`evaluate` returns two frames: `df_steps` (one row per timestep, one column per metric) and
`df_stats` (`mean`/`std`/`min`/`max`/`median` over those steps). A metric that cannot be computed
for a step — a diverged power flow, a missing `info` key — comes back as `NaN` rather than raising.


In [ ]:
METRIC_KEYS = [
    "max_line_loading_nminus0",
    "max_line_loading_nminus1",
    "max_line_loading_mean_nminus0",
    "max_line_loading_var_nminus0",
    "max_lines_overloaded_nminus0",
    "sum_line_loading_nminus0",
    "line_max_timestep_overload",
    "grid_timestep_overload",
    "open_busbar_couplers",
    "no_of_used_substations",
    "n_timesteps_substation_actions",
    "average_reward",
    "action_entropy",
]

step_results: dict[str, pd.DataFrame] = {}
stat_results: dict[str, pd.DataFrame] = {}

for name, actions in taken_actions.items():
    evaluator = EvaluateMetrics(copy.deepcopy(env_config), MetricRegistry.METRICS)
    step_results[name], stat_results[name] = evaluator.evaluate(
        actions,
        metric_keys=METRIC_KEYS,
        start_index=START_INDEX,
    )

step_results["Greedy"]


Stacking the `mean` row of every agent gives the comparison table: the greedy agent should show a
lower maximum line loading and fewer overloaded lines than doing nothing, at the price of using
substations and busbar couplers.


In [ ]:
comparison = pd.DataFrame({name: stats.loc["mean"] for name, stats in stat_results.items()})
comparison.index.name = "metric (mean over the episode)"
comparison.round(2)


## Per-metric plots

One figure per metric, one line per agent — the layout of the figures in `notebooks/plots`.
Loading metrics get the 100 % threshold drawn in.


In [ ]:
PERCENT_METRICS = {
    "max_line_loading_nminus0",
    "max_line_loading_nminus1",
    "max_line_loading_mean_nminus0",
}

# The PDFs in `notebooks/plots` are the paper's full-episode figures; re-running this notebook with
# a short N_STEPS would overwrite them, so saving is off unless you ask for it.
SAVE_FIGURES = False
OUTPUT_DIR = Path("plots")

colors = plt.get_cmap("tab10")
line_styles = ["-", "--", "-.", ":"]

for metric in METRIC_KEYS:
    plt.figure(figsize=(10, 2.5))
    for index, (name, df_steps) in enumerate(step_results.items()):
        plt.plot(
            df_steps.index + 1,
            df_steps[metric],
            label=name,
            color=colors(index % colors.N),
            linestyle=line_styles[index % len(line_styles)],
            marker="o",
            markersize=3,
        )
    if metric in PERCENT_METRICS:
        plt.axhline(y=100, color="black", linewidth=1.5, label="100 % threshold")

    plt.title(metric.replace("_", " ").title())
    plt.xlabel("Step")
    plt.ylabel(metric)
    plt.grid(visible=True)
    plt.legend()

    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(exist_ok=True)
        plt.savefig(OUTPUT_DIR / f"{metric}.pdf", format="pdf", bbox_inches="tight")
    plt.show()


## Adding your own metric

A metric is a function of a single `StepData` — the timestep index, the environment (and through
it the solved net and the action table), the action just applied, its reward and the `info` dict.
Register it with `MetricRegistry.add_metric` and it can be requested by name like any built-in one.
`overwrite=True` makes the cell re-runnable; without it, registering a known name raises `KeyError`.


In [ ]:
def line_loading_p95(step: StepData) -> float:
    """95th percentile of the N-0 line loadings, a less outlier-driven view than the maximum.

    :param step: the timestep context handed to every metric.
    :return: the 95th percentile of ``res_line.loading_percent`` in percent.
    """
    return float(step.env.net.res_line["loading_percent"].quantile(0.95))


MetricRegistry.add_metric("line_loading_p95", line_loading_p95, overwrite=True)

evaluator = EvaluateMetrics(copy.deepcopy(env_config), MetricRegistry.METRICS)
df_steps, df_stats = evaluator.evaluate(
    taken_actions["Greedy"],
    metric_keys=["max_line_loading_nminus0", "line_loading_p95"],
    start_index=START_INDEX,
)
df_stats.round(2)


Metrics that carry state across steps (the switching counters, the 8-hour busbar-coupler window)
keep it in `MetricRegistry.CACHE`, keyed per environment. `evaluate` clears that cache before every
run, so two evaluations never contaminate each other — but if you drive metrics manually, call
`MetricRegistry.reset()` yourself between runs.
